In [13]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import FloatType, DoubleType, IntegerType, BooleanType, TimestampType

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, OneHotEncoder
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.linalg import Vectors

import pandas as pd
import numpy as np


In [29]:
class CONFIG:
  def __init__(self) -> None:
    self.path = "/content/cleaned_data.csv"
    #self.standarized_path = "/content/normalized_data.csv"
    self.spark = SparkSession.builder.appName("test").getOrCreate()
  def read_csv(self, path):
    self.path = path
    df = self.spark.read.csv(self.path, header=True, inferSchema=True)
    return df
  #def standarize(self, df):

  def drop_null_horizontal(self, df):
        """Drops ROWS that have any null values."""
        df = df.dropna()
        return df

  def drop_null_vertical(self, df, columns):
      """Drops the specified COLUMNS from the dataframe."""
      df = df.drop(*columns)
      return df
  def cast_dtypes(df, cols, dtypes):
    for col, dtype in zip(cols, dtypes):
      df = df.withColumn(col, df[col].cast(dtype))
    return df

In [36]:
spark = CONFIG().spark
#na_cols = ["record_id","planting_date", "harvest_date", "expense_type", "expense_amount", "total_expenses", "profit", "fertilizer_used", "actual_yield", "expected_yield", "water_used","demand_level", "customer_type"]
df = CONFIG().read_csv(CONFIG().path)
df = CONFIG().drop_null_horizontal(df)
#df = CONFIG().drop_null_vertical(df, na_cols)
df = df.dropna()
df.show()

+---------+-----------+----+--------+-------------+-----------+---------+-----------------+-----------------+-------------------+------------------+
|record_id|  crop_name|year|   month|farm_location|region_type|area_size|actual_yield_tons| quantity_sold_kg|     unit_price_jod|   total_sales_jod|
+---------+-----------+----+--------+-------------+-----------+---------+-----------------+-----------------+-------------------+------------------+
|     6097|cauliflower|2022| January|       ajloun|    up-land|      4.0|             11.8|983.3333333333334|             0.2714|266.87666666666667|
|     6098|cauliflower|2022| January|       ajloun|    up-land|      4.0|             11.8|983.3333333333334|             0.2714|266.87666666666667|
|     6099|cauliflower|2022| January|       ajloun|    up-land|      4.0|             11.8|983.3333333333334|             0.2034|200.01000000000002|
|     6100|cauliflower|2022| January|       ajloun|    up-land|      4.0|             11.8|983.33333333333

In [42]:
def train_models(models, categorical_features, numerical_features, target, df):

    trainset, testset = df.randomSplit([0.8, 0.2], seed=42)

    indexer = StringIndexer(
        inputCols=categorical_features,
        outputCols=[f"{col}_index" for col in categorical_features]
    )

    '''encoder = OneHotEncoder(
        inputCols=[f"{col}_index" for col in categorical_features],
        outputCols=[f"{col}_ohe" for col in categorical_features]
    )'''

    assembler = VectorAssembler(
        inputCols=numerical_features + [f"{col}_index" for col in categorical_features],
        outputCol="features"
    )

    scaler = StandardScaler(
        inputCol="features",
        outputCol="scaled_features",
        withMean=True,
        withStd=True
    )

    stages = [indexer, assembler, scaler]

    evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction")
    best_R2 = 0
    best_model = {"name": None, "model": None}
    for model in models:
      model_name = model.__class__.__name__
      print(f"Training {model}")

      model.setParams(featuresCol="scaled_features", labelCol=target)

      pipeline = Pipeline(stages=stages + [model])

      fitted_model = pipeline.fit(trainset)
      predictions = fitted_model.transform(testset)

      predictions.show(2)

      r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})
      rmse = evaluator.evaluate(predictions, {evaluator.metricName: "rmse"})

      if r2 > best_R2:
          best_R2 = r2
          best_model["name"] = model_name
          best_model["model"] = fitted_model
      print(f"RMSE: {rmse:.2f} | R2: {r2:.2f}")
      print("-" * 30)
    #saving model
    model_name = best_model["name"]
    model_param = best_model["model"]
    model_param.write().overwrite().save(model_name)

In [43]:
cat_col = ["crop_name", "month", "farm_location", "region_Type"]
num_col = ["area_size","actual_yield_tons", "quantity_sold_kg", "unit_price_jod"]
target = "total_sales_jod"

models = [LinearRegression(featuresCol='scaled_features', labelCol=target),
          RandomForestRegressor(featuresCol='scaled_features', labelCol=target),
          GBTRegressor(featuresCol='scaled_features', labelCol=target)]

train_models(models, cat_col, num_col, target, df)

Training LinearRegression_150be7db6b0a
+---------+-----------+----+--------+-------------+-----------+---------+-----------------+-----------------+--------------+------------------+---------------+-----------+-------------------+-----------------+--------------------+--------------------+-------------------+
|record_id|  crop_name|year|   month|farm_location|region_type|area_size|actual_yield_tons| quantity_sold_kg|unit_price_jod|   total_sales_jod|crop_name_index|month_index|farm_location_index|region_Type_index|            features|     scaled_features|         prediction|
+---------+-----------+----+--------+-------------+-----------+---------+-----------------+-----------------+--------------+------------------+---------------+-----------+-------------------+-----------------+--------------------+--------------------+-------------------+
|     6099|cauliflower|2022| January|       ajloun|    up-land|      4.0|             11.8|983.3333333333334|        0.2034|200.01000000000002|  

In [45]:
!zip -r model.zip /content/GBTRegressor
from google.colab import files
files.download("model.zip")

updating: content/GBTRegressor/ (stored 0%)
updating: content/GBTRegressor/stages/ (stored 0%)
updating: content/GBTRegressor/stages/1_VectorAssembler_cca8db4f5068/ (stored 0%)
updating: content/GBTRegressor/stages/1_VectorAssembler_cca8db4f5068/metadata/ (stored 0%)
updating: content/GBTRegressor/stages/1_VectorAssembler_cca8db4f5068/metadata/_SUCCESS (stored 0%)
updating: content/GBTRegressor/stages/1_VectorAssembler_cca8db4f5068/metadata/._SUCCESS.crc (stored 0%)
updating: content/GBTRegressor/stages/1_VectorAssembler_cca8db4f5068/metadata/.part-00000-bd4b792c-549f-4b4c-ac97-1c7ad14c683d-c000.txt.crc (stored 0%)
updating: content/GBTRegressor/stages/1_VectorAssembler_cca8db4f5068/metadata/part-00000-bd4b792c-549f-4b4c-ac97-1c7ad14c683d-c000.txt (deflated 38%)
updating: content/GBTRegressor/stages/2_StandardScaler_263c55d26dd0/ (stored 0%)
updating: content/GBTRegressor/stages/2_StandardScaler_263c55d26dd0/data/ (stored 0%)
updating: content/GBTRegressor/stages/2_StandardScaler_263c5

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>